Objetivo del proyecto: Queremos clasificar una nueva imagen segun su categoria dentro del dataset.

- Origen de los datos: Kaggle
- Tarea: Convertir las imagenes en datos para poder trabajar con ellas
- Atributo a predecir: clase
- Tipos de atributo: Numericos y

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd # Crear y manipular nuestra tabla de datos, el DataFrame.
import numpy as np # Trabajar con las matrices de píxeles y calcular las medias RGB.
from pathlib import Path # Localizar las ocho carpetas y recorrer los archivos de imágenes.
from PIL import Image #	Abrir las imágenes y obtener sus dimensiones y píxeles.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, balanced_accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.base import clone

from features_hls import extract_hls_features

from 02_models import evaluate_model

RANDOM_STATE = 42

print("Entorn preparat.")

Entorn preparat.


In [11]:
data_dir = Path("../data")
rgb_path = Path("rgb_features.csv")

if rgb_path.exists():
    df = pd.read_csv(rgb_path)
    print("RGB: características cargadas desde CSV.")

else:
    categorias = []
    data = []

    for carpeta in data_dir.iterdir():
        if carpeta.is_dir():
            categorias.append(carpeta.name)

            for carpeta_interior in carpeta.iterdir():
                imagenes = list(carpeta_interior.iterdir())
                print("RGB: características cargadas desde CSV.")
            else:
                print("Treient dades de les imatges RGB:")
                print(carpeta_interior.name, ":", len(imagenes))
                
                for imagen in imagenes:
                    img = Image.open(imagen)

                    matriz = np.array(img.convert("RGB")) # (576, 720, 3) donde 3 es el RGB

                    #print(matriz.shape)
                    
                    # Seleccionamos los valores rojos de todos los píxeles
                    rojo = matriz[:, :, 0]
                    verde = matriz[:, :, 1]
                    azul = matriz[:, :, 2]

                    # calculamos la media porque si fuera la suma nos daria el total de rojo pero no queremos eso
                    rojo_medio = rojo.mean()
                    verde_medio = verde.mean()
                    azul_medio = azul.mean()

                    #print("Rojo medio:", rojo_medio)
                    #print("Verde medio:", verde_medio)
                    #print("Azul medio:", azul_medio)

                    rojo_std = rojo.std()
                    verde_std = verde.std()
                    azul_std = azul.std()

                    data.append({
                        "image_path": imagen.relative_to(data_dir).as_posix(),
                        "clase": carpeta.name,
                        "rojo_medio": rojo_medio,
                        "verde_medio": verde_medio,
                        "azul_medio": azul_medio,
                        "rojo_std": rojo_std,
                        "verde_std": verde_std,
                        "azul_std": azul_std
                    })
            df = pd.DataFrame(data)
            df.to_csv(rgb_path, index=False)
            print("RGB: características cargadas desde CSV.")

display(df.head())
df.describe()
#print(categorias)

hls_path = Path("hls_features.csv")

if hls_path.exists():
    df_hls = pd.read_csv(hls_path)
    print("HLS: características cargadas desde CSV.")
else:
    print("Treient dades de les imatges HLS:")
    data_hls = []

    for ruta in df["image_path"]:
        imagen = data_dir / ruta
        caracteristicas = extract_hls_features(imagen)
        data_hls.append(caracteristicas)
        
    df_hls = pd.DataFrame(data_hls)

    df_hls["image_path"] = df["image_path"].to_numpy()
    df_hls["clase"] = df["clase"].to_numpy()

    df_hls.to_csv(hls_path, index=False)
    print("HLS: características calculadas y guardadas.")

#print(df_hls.shape)
#print(df_hls.isna().sum())
#print(df_hls["clase"].value_counts())
#print(df_hls["image_path"].duplicated().sum())
#print(df["image_path"].equals(df_hls["image_path"]))

display(df_hls.head())
df_hls.describe()
#print(categorias)

# Entrenar modelos como Random Forest, KNN o SVM.
#
# Para cada imagen nueva, obtener:
# % de dyed-lifted-polyps
# % de dyed-resection-margins
# % de esophagitis
# % de normal-cecum
# % de normal-pylorus
# % de normal-z-line
# % de polyps
# % de ulcerative-colitis

# Queremos utilizar las curvas de aprendizaje y validación
# para encontrar el mejor modelo y sus hiperparámetros,
# evitando el sobreajuste (overfitting) y el subajuste (underfitting).

# Tendremos que evaluar el modelo usando Accuracy, F1-macro y AUC multiclase.

# Accuracy mide el porcentaje de imágenes clasificadas correctamente.

# F1-macro combina Precision y Recall, dando la misma importancia a cada clase.

# AUC-ROC multiclase mide la capacidad del modelo para distinguir
# cada clase del resto a través de diferentes umbrales.


#########################
# DEEP LEARNING

# CNN para que aprenda de las diferentes imagenes y sepa clasificarlas
# YOLO DETECTION para detectar polipos

# La CNN se evaluará con Accuracy, F1-macro y AUC multiclase.
# YOLO Detection se evaluará con mAP, Precision y Recall.


# Finalmente comparar CNN con los modelos de machine learning

RGB: características cargadas desde CSV.


,image_path,clase,rojo_medio,verde_medio,azul_medio,rojo_std,verde_std,azul_std
0,dyed-lifted-polyps/dyed-lifted-polyps/0053d7cd...,dyed-lifted-polyps,89.341975,78.168051,65.528431,72.205227,59.783039,55.773796
1,dyed-lifted-polyps/dyed-lifted-polyps/007d5aa7...,dyed-lifted-polyps,62.572789,73.194271,65.151705,56.633048,56.232007,52.351104
2,dyed-lifted-polyps/dyed-lifted-polyps/008aa3ed...,dyed-lifted-polyps,53.521316,69.116558,67.391351,58.343813,73.922001,74.345144
3,dyed-lifted-polyps/dyed-lifted-polyps/009171b0...,dyed-lifted-polyps,91.000080,75.910253,63.963023,84.849272,65.578390,58.403887
4,dyed-lifted-polyps/dyed-lifted-polyps/00cf9508...,dyed-lifted-polyps,71.580582,75.026114,72.785272,83.124550,82.302859,81.000454


HLS: características cargadas desde CSV.


,h_sin_mean,h_cos_mean,h_concentration,l_mean,l_std,s_mean,s_std,image_path,clase
0,0.222053,0.517819,0.563422,78.913788,56.352129,90.295486,67.323794,dyed-lifted-polyps/dyed-lifted-polyps/0053d7cd...,dyed-lifted-polyps
1,0.202805,0.169651,0.264408,65.497468,48.255186,75.223727,64.261989,dyed-lifted-polyps/dyed-lifted-polyps/007d5aa7...,dyed-lifted-polyps
2,0.242162,0.048731,0.247017,61.268041,65.396494,61.172613,55.117451,dyed-lifted-polyps/dyed-lifted-polyps/008aa3ed...,dyed-lifted-polyps
3,0.277804,0.564841,0.629460,78.867139,65.594705,91.486080,76.486448,dyed-lifted-polyps/dyed-lifted-polyps/009171b0...,dyed-lifted-polyps
4,0.309237,0.132344,0.336366,72.751608,81.817657,40.610928,50.815673,dyed-lifted-polyps/dyed-lifted-polyps/00cf9508...,dyed-lifted-polyps


,h_sin_mean,h_cos_mean,h_concentration,l_mean,l_std,s_mean,s_std
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,0.174164,0.726325,0.795736,100.122520,60.627734,99.642116,61.807609
std,0.147015,0.321526,0.222862,27.278854,7.863112,30.603414,14.763971
min,-0.514679,-0.539030,0.027580,25.477153,18.790425,25.865361,17.993021
25%,0.087763,0.703276,0.770563,80.576602,55.968119,74.850030,50.916728
50%,0.214320,0.842676,0.890654,92.809756,60.313824,95.349786,65.601409
75%,0.270222,0.958858,0.970766,120.294781,65.416216,124.023189,72.216384
max,0.624868,0.989165,0.992221,191.697056,95.411217,200.999701,99.831115


Modelos

In [ ]:

# Features
X_rgb = df.drop(["clase", "image_path"], axis=1)
X_hls = df_hls.drop(["clase", "image_path"], axis=1)

# Target (Ground Truth)
y = df["clase"]

# Comprobamos que las imágenes y clases coinciden
assert df["image_path"].equals(df_hls["image_path"])
assert df["clase"].equals(df_hls["clase"])

# Mismas particiones para RGB y HLS
X_rgb_train, X_rgb_test, X_hls_train, X_hls_test, y_train, y_test = train_test_split(
    X_rgb,
    X_hls,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# =========================
# RGB
# =========================

print("Dummy Classifier - RGB:")
evaluate_model(
    DummyClassifier(strategy="most_frequent"),
    X_rgb_train, y_train, X_rgb_test, y_test
)

print("Random Forest - RGB:")
evaluate_model(
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    X_rgb_train, y_train, X_rgb_test, y_test
)

# =========================
# HLS
# =========================

print("Dummy Classifier - HLS:")
evaluate_model(
    DummyClassifier(strategy="most_frequent"),
    X_hls_train, y_train, X_hls_test, y_test
)

print("Random Forest - HLS:")
evaluate_model(
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    X_hls_train, y_train, X_hls_test, y_test
)

Dummy Classifier - RGB:
Accuracy: 12.50%
Precision: 1.56%
Recall: 12.50%
F1-score: 2.78%


ValueError: Found input variables with inconsistent numbers of samples: [6400, 1600, 6400]